# Word embeddings: optional technical notes

These notes supplement the [lecture slides](dl-representations_simple.ipynb). They develop count weighting and compression, parameter initialization, and the mathematical connection between prediction and co-occurrence statistics.

The worked examples below use a separate **12-sentence toy corpus**.

## A corpus for the worked examples

<img src="../img/word2vec_2027/corpus_pairs.svg" alt="Twelve short sentences and the ordered pairs extracted with a symmetric radius-two window" width="1000">

A centre word is also called a target word. Here we say centre and context to keep the two roles clear. Twelve three-token sentences give 36 tokens and 72 ordered centre–context pairs. The window never crosses a sentence boundary. For example, the first sentence contributes (apples, are), (apples, tasty), (are, apples), (are, tasty), (tasty, apples), (tasty, are). Repeated sentences contribute repeated pairs. The code generates pairs directly from tokens and only then accumulates a count matrix for inspection.

## Count weighting and compression

**Positive pointwise mutual information (PPMI)** weights how strongly a word and a context are associated. A truncated **singular value decomposition (SVD)** can then produce dense word vectors from that weighted matrix.

<img src="../img/word2vec_2027/counts.svg" alt="Word-context count matrix for the toy corpus" width="1000">

For total pair count N, row count C_w and column count C_c, PMI(w,c) = log(C_wc N / (C_w C_c)); PPMI clips negative PMI to zero and assigns zero to unobserved pairs. Here N=72, C_apples=8, C_are=24 and C_tasty=16. Both pairs have count 4, but PMI(apples,are)=log(1.5)=0.405 and PMI(apples,tasty)=log(2.25)=0.811. The less frequent context is more informative. The diagram shows a real rank-two truncated SVD of this corpus's PPMI matrix, using U_2 Sigma_2 as the word vectors. Other scalings of the singular values are possible; the axes have no intrinsic names. This is still a count-based method even though its final vectors are dense. Source: [Levy & Goldberg (2014)](https://papers.nips.cc/paper_files/paper/2014/file/b78666971ceae55a8e87efb7cbfd9ad4-Paper.pdf).

<img src="../img/word2vec_2027/count_to_dense.svg" alt="Example pair counts, their PPMI weights, and a rank-two SVD representation" width="1000">

## Network parameters

Let $E\in\mathbb{R}^{|V|\times d}$ contain the input word vectors and $O\in\mathbb{R}^{|V|\times d}$ contain output vectors. The output weight matrix in the lecture network is $O^\top$ if vectors multiply weights as rows.

For a centre word $w$ and a context word $c$, the score is $e_w^\top o_c$. These are two sets of network weights with different roles. A common downstream choice uses the rows of $E$ as the static word embeddings.

## Initialization

The original C InitNet initializes syn0 (E) to pseudorandom values approximately uniform on [-0.5/d, 0.5/d), and syn1neg (O for negative sampling) to zero. This is an implementation choice, not a requirement that all word2vec implementations initialize identically. The figure shows the first three coordinates of the eight-dimensional teaching run. At this initialization every score is zero, so full softmax predicts 1/|V| for every word and sigmoid predicts 1/2 for each pair. Word-frequency counts are still needed for the vocabulary, subsampling and noise distribution; they do not provide pretrained embedding coordinates. Source: [word2vec.c, InitNet](https://github.com/tmikolov/word2vec/blob/master/word2vec.c).

<img src="../img/word2vec_2027/initialization.svg" alt="The original initialization recipe illustrated with the first three coordinates of the toy experiment" width="1000">

With a bilinear score $s=e^\top o$, a logistic-loss contribution has gradients $\nabla_e L=(\sigma(s)-y)o$ and $\nabla_o L=(\sigma(s)-y)e$. If $E$ is random and every row of $O$ is zero, the first update can change $O$ while the gradient for $E$ is zero. This is a consequence of that initialization, not a general property of every training step or every implementation. If both sets of weights were zero, both gradients would be zero.

The input vectors start without semantic information. Co-occurrence counts are not their initial coordinates.

## Full softmax and prediction loss

$$p(c\mid w)=\frac{\exp(o_c^\top e_w)}{\sum_{j\in V}\exp(o_j^\top e_w)},\qquad L=-\log p(c\mid w).$$

At initialization p=1/7 and loss=log(7)=1.946. The derivative with respect to logit c is p(c|w)-1[c=tasty]. It is -6/7 for tasty and +1/7 for each other word. This step pushes tasty up relative to the alternatives; other real contexts get their own positive training examples. Full softmax computes scores and gradients for every output row, costing O(|V|d) per pair. Repeated training shares each vector across many pairs.

<img src="../img/word2vec_2027/softmax.svg" alt="Uniform full-softmax probabilities at initialization for the seven-word toy vocabulary" width="1000">

## Skip-gram with negative sampling (SGNS)

$$L=-\log\sigma(e_w^\top o_c)-\sum_{i=1}^{k}\log\sigma(-e_w^\top o_{n_i}).$$

Negative sampling replaces the normalized full-softmax objective with binary discrimination between observed pairs and noise pairs. Here k=2, centre apples is fixed, tasty is the positive context, and furry and rabbits are illustrative noise samples. The sigmoid scores are not a normalized context-word distribution. Noise is drawn from q(c) proportional to token frequency(c)^(3/4) in the original word2vec recipe. A sampled pair is not known to be semantically false or impossible: it can also occur elsewhere in the corpus. Some implementations reject a sample equal to the current positive context. The teaching code uses independent draws with replacement, including possible collisions, so that its expected objective is particularly simple. The cost is O((k+1)d) per observed pair rather than O(|V|d). Source: [Mikolov et al. (2013), Distributed Representations](https://arxiv.org/abs/1310.4546).

<img src="../img/word2vec_2027/negative_sampling.svg" alt="One observed pair and two independently sampled noise pairs" width="1000">

## A worked stochastic gradient descent (SGD) step

This separate hand-worked example has d=2, k=1 and learning rate eta=0.2; it is one step after initialization, not the first step of the eight-dimensional training run. Start with e=(1,0), o+=(0,1), o-=(0,-1). Both logits are zero, so sigmoid scores are 0.5 and errors are g+=-0.5 and g-=+0.5. The input gradient is g+ o+ + g- o- = (0,-1). Output gradients are g+ e=(-0.5,0) and g- e=(0.5,0). Simultaneously updating from the old values gives e'=(1,0.2), o+'=(0.1,1), o-'=(-0.1,-1). New dot products are +0.3 and -0.3, new sigmoid scores are 0.57444 and 0.42556, and the loss falls from 2 log(2)=1.38629 to 2 log(1+exp(-0.3))=1.10871. This directly changes the input–output dot products; it does not directly pull the two input rows E[apples] and E[tasty] together.

<img src="../img/word2vec_2027/gradient_step.svg" alt="A simultaneous SGD step changes input and output vectors and reduces the two-pair loss" width="1000">

In [ ]:
# Reproducible toy skip-gram with negative sampling (SGNS) experiment; rebuild the figures with:
# python scripts/build_word2vec_figures.py  (from the repository root)
import numpy as np

vocabulary = ["apples", "oranges", "rabbits", "hamsters", "are", "tasty", "furry"]
word_id = {word: i for i, word in enumerate(vocabulary)}
corpus = (["apples are tasty"] * 4 + ["oranges are tasty"] * 4
          + ["rabbits are furry"] * 2 + ["hamsters are furry"] * 2)
window_radius = 2
pairs = []
token_counts = np.zeros(len(vocabulary), dtype=int)
for sentence in corpus:
    ids = [word_id[word] for word in sentence.split()]
    np.add.at(token_counts, ids, 1)
    for position, centre in enumerate(ids):
        for context_position in range(max(0, position - window_radius),
                                      min(len(ids), position + window_radius + 1)):
            if position != context_position:
                pairs.append((centre, ids[context_position]))
# Sorting only fixes a reproducible starting order before epoch-wise shuffling.
pairs = np.array(sorted(pairs), dtype=int)
# C is for analysis and figures, not a prerequisite for the training updates.
C = np.zeros((len(vocabulary), len(vocabulary)), dtype=int)
np.add.at(C, (pairs[:, 0], pairs[:, 1]), 1)

rng = np.random.default_rng(7)
dimensions, negative_samples, epochs = 8, 2, 400
E = rng.uniform(-0.5 / dimensions, 0.5 / dimensions,
                (len(vocabulary), dimensions))
O = np.zeros_like(E)
initial_E = E.copy()
noise_distribution = token_counts.astype(float) ** 0.75
noise_distribution /= noise_distribution.sum()

def expected_sgns_loss(input_table, output_table):
    scores = input_table @ output_table.T
    positive = C * np.logaddexp(0, -scores)
    noise_weights = negative_samples * C.sum(axis=1)[:, None] * noise_distribution
    negative = noise_weights * np.logaddexp(0, scores)
    return (positive + negative).sum() / len(pairs)

loss_history = [expected_sgns_loss(E, O)]
for epoch in range(epochs):
    learning_rate = 0.06 * (1 - epoch / epochs) + 0.005
    for pair_index in rng.permutation(len(pairs)):
        centre, context = pairs[pair_index]
        # Independent noise draws with replacement, including possible collisions.
        output_rows = np.r_[context, rng.choice(len(vocabulary), negative_samples,
                                                p=noise_distribution)]
        labels = np.r_[1.0, np.zeros(negative_samples)]
        old_input = E[centre].copy()
        old_outputs = O[output_rows].copy()
        scores = old_outputs @ old_input
        errors = 1 / (1 + np.exp(-scores)) - labels
        E[centre] -= learning_rate * (errors @ old_outputs)
        # add.at correctly accumulates updates when a sampled row is repeated.
        np.add.at(O, output_rows, -learning_rate * errors[:, None] * old_input)
    loss_history.append(expected_sgns_loss(E, O))

def cosine_matrix(vectors):
    unit_vectors = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)
    return unit_vectors @ unit_vectors.T

initial_cosines = cosine_matrix(initial_E[:4])
trained_cosines = cosine_matrix(E[:4])

## Measured toy training results

The toy run uses eight dimensions for seven vocabulary items to illustrate learning; it is not a compression example. Real word2vec vocabularies are usually much larger than the embedding dimension. The two pairs with identical context distributions end up with high input–input cosine similarity even though the words in each pair never directly co-occur. Read the measured values in the heatmaps; similarity is not guaranteed to match a hand-labelled ontology or every random run. The loss curve can fluctuate because each update uses sampled negatives. More dimensions and more epochs are not automatically better on real data. Here the vocabulary and corpus are deliberately tiny enough to inspect. All values and figures are regenerated by scripts/build_word2vec_figures.py from the tagged code cell above.

<img src="../img/word2vec_2027/training_results.svg" alt="Cosines before and after training and the expected SGNS loss" width="1000">

## The connection to matrix factorization

Pointwise mutual information was introduced in the count-based section. Let C_wc be an observed pair count, C_w the centre count over pairs, q(c) the noise distribution and k the number of negatives per positive. The expected contribution of one score s is C_wc log sigma(s) + k C_w q(c) log sigma(-s). If each score can vary independently, differentiating yields s*=log[C_wc/(k C_w q(c))] = log[P_D(c|w)/(k q(c))]. Taking q(c)=P_D(c), the empirical context marginal, gives PMI(w,c)-log k. For zero-count pairs the unconstrained optimum tends to negative infinity. A finite-dimensional product E O^T couples scores and approximates them under the SGNS weighting; it does not independently attain every optimum. The usual q proportional to frequency^0.75 changes the shift to depend on c. Input and output factorizations need not give identical input cosine geometry even when score matrices agree. Source: [Levy & Goldberg (2014), Neural Word Embedding as Implicit Matrix Factorization](https://papers.nips.cc/paper_files/paper/2014/file/b78666971ceae55a8e87efb7cbfd9ad4-Paper.pdf).

<img src="../img/word2vec_2027/implicit_matrix.svg" alt="Input and output matrices define a low-rank score matrix with a shifted-PMI connection under stated assumptions" width="1000">

## Global Vectors (GloVe)

GloVe minimizes sum over observed pairs f(C_wc)(e_w^T o_c+b_w+b_c-log C_wc)^2, with a weighting function that limits the influence of very frequent co-occurrences. PPMI+SVD instead minimizes a low-rank reconstruction error for its transformed matrix. SGNS weights the positive and negative logistic terms according to pair and noise frequencies. Choices of window, weighting, dimension and evaluation matter; no method wins merely because it is called neural. Source: [GloVe paper](https://aclanthology.org/D14-1162/) and [Levy et al. (2015)](https://aclanthology.org/Q15-1016/).

## Document counts and TF–IDF

For scikit-learn's default smoothed IDF, idf(w)=log((1+N)/(1+df(w)))+1; multiply by term counts and apply the configured row normalization (L2 by default). Document frequency counts documents containing w, not the number of occurrences. This document representation is distinct from the word-context rows used in this lecture.

## References

- [Original word2vec implementation](https://github.com/tmikolov/word2vec/blob/master/word2vec.c), especially `InitNet` and `TrainModelThread`
- Mikolov et al. (2013), [Distributed Representations of Words and Phrases](https://arxiv.org/abs/1310.4546)
- Levy & Goldberg (2014), [Neural Word Embedding as Implicit Matrix Factorization](https://papers.nips.cc/paper_files/paper/2014/file/b78666971ceae55a8e87efb7cbfd9ad4-Paper.pdf)
- Pennington et al. (2014), [GloVe](https://aclanthology.org/D14-1162/)
- Levy et al. (2015), [Improving Distributional Similarity](https://aclanthology.org/Q15-1016/)